### Limpeza, Tratamento e Feature Engineering

#### Resumo Geral

Este notebook tem como objetivo analisar os dados de criminalidade envolvendo veículos no estado de São Paulo, buscando identificar padrões, inconsistências e necessidades de tratamento e padronização dos dados.

Nele constam:

- leitura inicial dos dados
- padronização de nomenclaturas
- identificação de valores nulos e campos prioritários
- tratamento e conversão de tipos de dados
- criação de features temporais e categóricas e flags
- preparação inicial para análises estatísticas e geoespaciais

In [1]:
from pathlib import Path
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

ROOT = Path().resolve().parent
sys.path.append(str(ROOT))

In [ ]:
## Imports
from pyspark.sql.functions import col, trim, lower
from src.pyspark.extract import extract_veiculos
from src.pyspark.transform import transform_veiculos ### Importa as funções criadas ao decorrer do texto

In [3]:
### Ler um dos arquivos e validar schema inicial
file_path = "../data/criminalidade/raw/VeiculosSubtraidos_2026.xlsx"

df_raw = extract_veiculos(file_path, 2026)
df_raw.printSchema()

root
 |-- ID_DELEGACIA: string (nullable = true)
 |-- NOME_DEPARTAMENTO: string (nullable = true)
 |-- NOME_SECCIONAL: string (nullable = true)
 |-- NOME_DELEGACIA: string (nullable = true)
 |-- NOME_MUNICIPIO: string (nullable = true)
 |-- ANO_BO: string (nullable = true)
 |-- NUM_BO: string (nullable = true)
 |-- VERSAO: string (nullable = true)
 |-- NOME_DEPARTAMENTO_CIRC: string (nullable = true)
 |-- NOME_SECCIONAL_CIRC: string (nullable = true)
 |-- NOME_DELEGACIA_CIRC: string (nullable = true)
 |-- NOME_MUNICIPIO_CIRC: string (nullable = true)
 |-- DATA_OCORRENCIA_BO: string (nullable = true)
 |-- HORA_OCORRENCIA: string (nullable = true)
 |-- DESCRICAO_APRESENTACAO: string (nullable = true)
 |-- DATAHORA_REGISTRO_BO: string (nullable = true)
 |-- DATA_COMUNICACAO_BO: string (nullable = true)
 |-- DATAHORA_IMPRESSAO_BO: string (nullable = true)
 |-- DESCR_PERIODO: string (nullable = true)
 |-- AUTORIA_BO: string (nullable = true)
 |-- FLAG_INTOLERANCIA: string (nullable = true)


#### Colunas despadronizadas
As colunas estão em um formato não padronizado com espaços a mais e em maíusculo.
Criada a função normalize_columns para facilitar a análise

#### Pensamento geral do Dataset
Os dados parecem ter mais de uma linha para cada registro, com o mesmo veículo sendo registro como Furto e posteriormente Recuperação por exmeplo. Precisaremos deduplicar as linhas antes de analisar

In [8]:
df = transform_veiculos(df_raw)
df.printSchema()
df.show(n=10, truncate=False, vertical=False)

root
 |-- id_delegacia: string (nullable = true)
 |-- nome_departamento: string (nullable = true)
 |-- nome_seccional: string (nullable = true)
 |-- nome_delegacia: string (nullable = true)
 |-- nome_municipio: string (nullable = true)
 |-- ano_bo: string (nullable = true)
 |-- num_bo: string (nullable = true)
 |-- versao: string (nullable = true)
 |-- nome_departamento_circ: string (nullable = true)
 |-- nome_seccional_circ: string (nullable = true)
 |-- nome_delegacia_circ: string (nullable = true)
 |-- nome_municipio_circ: string (nullable = true)
 |-- data_ocorrencia_bo: string (nullable = true)
 |-- hora_ocorrencia: string (nullable = true)
 |-- descricao_apresentacao: string (nullable = true)
 |-- datahora_registro_bo: string (nullable = true)
 |-- data_comunicacao_bo: string (nullable = true)
 |-- datahora_impressao_bo: string (nullable = true)
 |-- descr_periodo: string (nullable = true)
 |-- autoria_bo: string (nullable = true)
 |-- flag_intolerancia: string (nullable = true)


In [7]:
total_count = df.count()

for field in df.columns:

    null_count = df.filter(
        col(field).isNull() |
        (trim(col(field).cast("string")) == "") |
        (lower(trim(col(field).cast("string"))) == "nan")
    ).count()

    print(
        f"{field}: {null_count} nulls "
        f"({round(null_count / total_count * 100, 2)}%)"
    )

id_delegacia: 0 nulls (0.0%)
nome_departamento: 0 nulls (0.0%)
nome_seccional: 0 nulls (0.0%)
nome_delegacia: 0 nulls (0.0%)
nome_municipio: 0 nulls (0.0%)
ano_bo: 0 nulls (0.0%)
num_bo: 0 nulls (0.0%)
versao: 0 nulls (0.0%)
nome_departamento_circ: 1 nulls (0.0%)
nome_seccional_circ: 1 nulls (0.0%)
nome_delegacia_circ: 1 nulls (0.0%)
nome_municipio_circ: 1 nulls (0.0%)
data_ocorrencia_bo: 0 nulls (0.0%)
hora_ocorrencia: 3947 nulls (11.91%)
descricao_apresentacao: 0 nulls (0.0%)
datahora_registro_bo: 0 nulls (0.0%)
data_comunicacao_bo: 0 nulls (0.0%)
datahora_impressao_bo: 0 nulls (0.0%)
descr_periodo: 29185 nulls (88.09%)
autoria_bo: 0 nulls (0.0%)
flag_intolerancia: 0 nulls (0.0%)
tipo_intolerancia: 33132 nulls (100.0%)
flag_flagrante: 0 nulls (0.0%)
flag_status: 0 nulls (0.0%)
desc_lei: 0 nulls (0.0%)
flag_ato_infracional: 0 nulls (0.0%)
rubrica: 0 nulls (0.0%)
descr_conduta: 12275 nulls (37.05%)
desdobramento: 31710 nulls (95.71%)
circunstancia: 26792 nulls (80.86%)
descr_tipolocal:

# Limpeza Inicial dos Dados
Temos poucos nulls no geral, a maioria gerenciável. Devemos seguir com plano:

#### Colunas removidas
-- Sem muito valor para a análise
- `tipo_intolerancia`
- `flag_intolerancia`
- `flag_ato_infracional`
- `logradouro_versao`
- `desc_lei`
- `descricao_apresentacao`
- `placa_veiculo` -- Após deduplicação \

-- Proporção de nullos muito considerável:
- `desdobramentos`
- `ciscunstância`
#### Colunas com tratamento obrigatório

- `hora_ocorrencia`
  - conversão para hora
  - criação de períodos do dia

- `descr_periodo`
  - recriação a partir da hora da ocorrência
  - validação pós recriação
  - remoção após validação

- `bairro`
  - padronização textual
  - possível preenchimento via coordenadas

- `latitude` e `longitude`
  - conversão numérica
  - validação geográfica (coordenadas reais para a cidade de São Paulo)

- `descr_conduta`, `circunstancia`, `desdobramento`
  - padronização e agrupamento categórico

#### Tratamentos gerais
- normalização textual
- conversão de datas
- conversão de tipos
- remoção de `"nan"` e strings vazias
- deduplicação de ocorrências

In [6]:
from src.pyspark.transform import transform_veiculos ### Importa as funções criadas ao decorrer do texto
df = transform_veiculos(df_raw)
df.printSchema()
df.show(n=5, truncate=False, vertical=False)

root
 |-- id_delegacia: integer (nullable = true)
 |-- nome_seccional: string (nullable = true)
 |-- nome_delegacia: string (nullable = true)
 |-- nome_municipio: string (nullable = true)
 |-- ano_bo: integer (nullable = true)
 |-- num_bo: string (nullable = true)
 |-- versao: integer (nullable = true)
 |-- nome_departamento_circ: string (nullable = true)
 |-- nome_seccional_circ: string (nullable = true)
 |-- nome_delegacia_circ: string (nullable = true)
 |-- nome_municipio_circ: string (nullable = true)
 |-- data_ocorrencia_bo: date (nullable = true)
 |-- hora_ocorrencia: string (nullable = true)
 |-- descricao_apresentacao: string (nullable = true)
 |-- datahora_registro_bo: timestamp (nullable = true)
 |-- data_comunicacao_bo: date (nullable = true)
 |-- datahora_impressao_bo: timestamp (nullable = true)
 |-- descr_periodo: string (nullable = true)
 |-- autoria_bo: string (nullable = true)
 |-- flag_flagrante: string (nullable = true)
 |-- flag_status: string (nullable = true)
 |--

## Funções de Transformação Utilizadas

- `normalize_columns()`
  - normalização dos nomes das colunas

- `trim_string_columns()`
  - remoção de espaços extras em colunas string

- `clean_null_strings()`
  - substituição de `"nan"`, `"null"` e strings vazias por valores nulos

- `drop_unnecessary_columns()`
  - remoção de colunas sem valor analítico

- `uppercase_string_columns()`
  - padronização textual em maiúsculo

- `cast_date_columns()`
  - conversão de colunas para tipo date

- `cast_timestamp_columns()`
  - conversão de colunas para tipo timestamp

- `cast_integer_columns()`
  - conversão de colunas numéricas inteiras

- `cast_time_columns()`
  - padronização de colunas de horário

- `cast_coordinate_columns()`
  - limpeza e conversão de latitude e longitude

- `create_time_features()`
  - criar colunas significativas em relação as principais informações temporais
   
- `create_periodo_dia`
  - criar uma coluna de período do dia em que a ocorrência acontece em substituição da descr_periodo (88% nulla)

In [21]:
columns_categorize = [
    #"bairro",
    "descr_conduta",
    "rubrica",
    "descr_ocorrencia_veiculo",
    "descr_tipo_veiculo",
    # "descr_marca_veiculo",
    "descr_tipolocal"
]

for column_name in columns_categorize:

    print(f"\n### {column_name} ###")

    (
        df.groupBy(column_name)
        .count()
        .orderBy(col("count").desc())
        .show(15, truncate=False, vertical=True) ## Mudar o número no início do parentêses para visualizar mais resultados
    )


### descr_conduta ###
-RECORD 0----------------------------------------
 descr_conduta | VEÍCULO                         
 count         | 19150                           
-RECORD 1----------------------------------------
 descr_conduta | NULL                            
 count         | 12275                           
-RECORD 2----------------------------------------
 descr_conduta | OUTROS                          
 count         | 577                             
-RECORD 3----------------------------------------
 descr_conduta | CARGA                           
 count         | 402                             
-RECORD 4----------------------------------------
 descr_conduta | RESIDÊNCIA                      
 count         | 212                             
-RECORD 5----------------------------------------
 descr_conduta | TRANSEUNTE                      
 count         | 131                             
-RECORD 6----------------------------------------
 descr_conduta | INTERIOR D

### Criação de Variáveis Analíticas

Nesta etapa foram criadas novas colunas categóricas e temporais para facilitar a análise exploratória, visualização e interpretação dos dados.

#### Categorias criadas

- `categoria_conduta`
  - agrupa a conduta da ocorrência em categorias mais interpretáveis

- `categoria_rubrica`
  - classifica a natureza da ocorrência, como furto, roubo, recuperação, receptação e outros

- `categoria_ocorrencia_veiculo`
  - padroniza a situação do veículo como furtado, roubado ou recuperado

- `categoria_tipo_veiculo`
  - agrupa os tipos de veículos em categorias como leve, motocicleta, pesado, transporte público e outros

- `categoria_tipolocal`
  - agrupa o tipo de local da ocorrência, como via pública, residencial, comercial, estacionamento e outros

#### Variáveis temporais criadas

- `hora_ocorrencia_num`
  - extrai a hora da ocorrência em formato numérico

- `periodo_dia`
  - classifica a ocorrência em madrugada, manhã, tarde ou noite
  - substitui a variável original `descr_periodo`, que possui alta quantidade de valores nulos

#### Objetivo da transformação

Essas variáveis tornam os dados mais adequados para:

- análises geoespaciais
- mapas temáticos
- análise temporal
- comparação entre tipos de ocorrência
- agregações por bairro, período e tipo de veículo
- modelos estatísticos futuros